# 1. syncio(동기) vs Asyncio(비동기)

In [6]:
import time
import asyncio

# ==========================================
# 1. 동기 방식 (Synchronous / Blocking)
# ==========================================
def sync_task(task_id: int):
    """
    동기 작업은 time.sleep()을 만나면 스레드 전체의 실행을 멈춥니다(Blocking).
    하나의 작업이 완전히 끝나야만 다음 작업이 시작될 수 있습니다.
    """
    print(f"🔒 [동기] 작업 {task_id} 시작")
    time.sleep(2)  # 2초간 스레드 전체가 대기 상태에 빠짐
    print(f"🔓 [동기] 작업 {task_id} 완료")

def run_sync():
    print("--- 동기(Sync) 실행 시작 ---")
    start_time = time.perf_counter()

    # 3개의 작업을 순차적으로 실행
    for i in range(1, 4):
        sync_task(i)

    elapsed_time = time.perf_counter() - start_time
    # 2초 * 3개 = 약 6초 소요
    print(f"⏱️ 동기 총 소요 시간: {elapsed_time:.2f}초\n") 


# ==========================================
# 2. 비동기 방식 (Asynchronous / Non-blocking)
# ==========================================
async def async_task(task_id: int):
    """
    비동기 작업은 await asyncio.sleep()을 만나면 대기하는 동안 
    이벤트 루프에 제어권을 양보(Yield)합니다(Non-blocking).
    기다리는 동안 멈춰있지 않고 다른 작업을 병행해서 처리할 수 있습니다.
    """
    print(f"🚀 [비동기] 작업 {task_id} 시작")
    await asyncio.sleep(2)  # 2초간 대기하며 다른 작업 실행을 허용
    print(f"🏁 [비동기] 작업 {task_id} 완료")

async def run_async():
    print("--- 비동기(Async) 실행 시작 ---")
    start_time = time.perf_counter()

    # Python 3.11 이상에서 지원하는 TaskGroup 활용 (안전한 동시성 관리)
    async with asyncio.TaskGroup() as tg:
        # 3개의 작업을 이벤트 루프에 등록하여 동시에 실행
        for i in range(1, 4):
            tg.create_task(async_task(i))

    elapsed_time = time.perf_counter() - start_time
    # 3개의 작업이 2초를 동시에 대기하므로 = 약 2초 소요
    print(f"⏱️ 비동기 총 소요 시간: {elapsed_time:.2f}초\n")


# ==========================================
# 실행 진입점 (Entry Point)
# ==========================================
if __name__ == "__main__":
    # 동기 코드 실행
    run_sync()
    
    # 비동기 코드 실행 (이벤트 루프 생성 및 실행)
    await run_async()

--- 동기(Sync) 실행 시작 ---
🔒 [동기] 작업 1 시작
🔓 [동기] 작업 1 완료
🔒 [동기] 작업 2 시작
🔓 [동기] 작업 2 완료
🔒 [동기] 작업 3 시작
🔓 [동기] 작업 3 완료
⏱️ 동기 총 소요 시간: 6.00초

--- 비동기(Async) 실행 시작 ---
🚀 [비동기] 작업 1 시작
🚀 [비동기] 작업 2 시작
🚀 [비동기] 작업 3 시작
🏁 [비동기] 작업 1 완료
🏁 [비동기] 작업 2 완료
🏁 [비동기] 작업 3 완료
⏱️ 비동기 총 소요 시간: 2.02초



In [10]:
# 코루틴 객체, 태스크 (Task), 퓨처 (Future)의 구분
import asyncio
import time

async def fetch_data(task_id: int) -> str:
    """비동기 데이터 처리를 시뮬레이션하는 코루틴 함수"""
    print(f"📡 [실행] fetch_data({task_id}) 함수 내부 코드가 실행을 시작했습니다.")
    await asyncio.sleep(1)  # 제어권 양보
    return f"데이터 {task_id}"

async def main():
    # -------------------------------------------------------------
    # 1. 코루틴 객체 (Coroutine Object) : 실행 계획서
    # -------------------------------------------------------------
    print("=== [1] 코루틴 객체 확인 ===")
    coro = fetch_data(1)
    
    print(f"• 객체 타입: {type(coro)}")  # <class 'coroutine'>
    print("• 💡 호출은 했지만 아직 함수 내부의 '[실행]' 로그가 찍히지 않았습니다. (실행 대기 상태)")
    
    # await를 만나야 이벤트 루프가 이 계획서를 읽고 실행합니다.
    result = await coro 
    print(f"• 결과 회수: {result}\n")


    # -------------------------------------------------------------
    # 2. 태스크 (Task) : 이벤트 루프에 등록된 실행 단위
    # -------------------------------------------------------------
    print("=== [2] 태스크 (Task) 확인 ===")
    # create_task를 호출하는 순간, 백그라운드에서 즉시 실행 스케줄링이 시작됩니다.
    task = asyncio.create_task(fetch_data(2)) 
    
    print(f"• 객체 타입: {type(task)}")  # <class '_asyncio.Task'>
    # ★ 표에 있던 이론 검증: Task는 Future의 하위 클래스(자식)인가?
    print(f"• 💡 Task가 Future의 하위 클래스인가요? -> {isinstance(task, asyncio.Future)}") # True
    print(f"• 현재 Task 완료 여부: {task.done()}")  # False (아직 1초가 안 지나서 작업 중)
    
    print("• ⏳ 메인 루프에서 다른 작업(0.5초 대기)을 수행하며 멈추지 않고 진행합니다...")
    await asyncio.sleep(0.5)  
    
    # 나중에 작업이 완료되면 결과를 회수합니다.
    result2 = await task      
    print(f"• 완료 후 Task 완료 여부: {task.done()}")  # True
    print(f"• 결과 회수: {result2}\n")


    # -------------------------------------------------------------
    # 3. 퓨처 (Future) : 미래에 결과가 채워질 빈 상자
    # -------------------------------------------------------------
    print("=== [3] 퓨처 (Future) 확인 ===")
    # 개발자가 직접 제어하는 저수준 빈 상자 객체 생성
    loop = asyncio.get_running_loop()
    future = loop.create_future()  # 또는 asyncio.Future()
    
    print(f"• 객체 타입: {type(future)}")  # <class '_asyncio.Future'>
    print(f"• 현재 Future 완료 여부: {future.done()}")  # False (아직 상자가 비어있음)
    
    # 0.5초 뒤에 백그라운드에서 빈 상자(Future)에 데이터를 채워주는 비동기 함수 정의 및 실행
    async def fill_box_later(fut: asyncio.Future):
        await asyncio.sleep(0.5)
        print("🎁 [Future] 빈 상자에 데이터를 채워 넣습니다! (set_result 호출)")
        fut.set_result("미래에서 도착한 결과물")  # 빈 상자에 값을 채우고 상태를 완료(done)로 변경

    asyncio.create_task(fill_box_later(future))

    # Future 상자에 값이 채워질 때까지 기다렸다가 결과 회수
    print("• ⏳ Future 상자에 결과가 채워지기를 (await로) 기다리는 중...")
    result3 = await future
    print(f"• 완료 후 Future 완료 여부: {future.done()}")  # True
    print(f"• 결과 회수: {result3}\n")


if __name__ == "__main__":
    start_time = time.perf_counter()
    
    # 이벤트 루프 구동
    await main()
    
    print(f"⏱️ 총 소요 시간: {time.perf_counter() - start_time:.2f}초")

=== [1] 코루틴 객체 확인 ===
• 객체 타입: <class 'coroutine'>
• 💡 호출은 했지만 아직 함수 내부의 '[실행]' 로그가 찍히지 않았습니다. (실행 대기 상태)
📡 [실행] fetch_data(1) 함수 내부 코드가 실행을 시작했습니다.
• 결과 회수: 데이터 1

=== [2] 태스크 (Task) 확인 ===
• 객체 타입: <class '_asyncio.Task'>
• 💡 Task가 Future의 하위 클래스인가요? -> True
• 현재 Task 완료 여부: False
• ⏳ 메인 루프에서 다른 작업(0.5초 대기)을 수행하며 멈추지 않고 진행합니다...
📡 [실행] fetch_data(2) 함수 내부 코드가 실행을 시작했습니다.
• 완료 후 Task 완료 여부: True
• 결과 회수: 데이터 2

=== [3] 퓨처 (Future) 확인 ===
• 객체 타입: <class '_asyncio.Future'>
• 현재 Future 완료 여부: False
• ⏳ Future 상자에 결과가 채워지기를 (await로) 기다리는 중...
🎁 [Future] 빈 상자에 데이터를 채워 넣습니다! (set_result 호출)
• 완료 후 Future 완료 여부: True
• 결과 회수: 미래에서 도착한 결과물

⏱️ 총 소요 시간: 2.54초


In [13]:
# 구조적 동시성 (TaskGroup): Python 3.11부터 도입된 `TaskGroup`은 자식 Task 중 하나라도 예외가 발생하면 나머지를 안전하게 **자동 취소**하여 좀비 Task를 방지합니다.

import asyncio

async def fetch_data(task_id: int):
    print(f"▶️ [시작] 작업 {task_id} 실행 중...")
    try:
        await asyncio.sleep(1) # 1초 대기 (네트워크 요청 시뮬레이션)

        if task_id == 2:
            print(f"💥 [에러] 작업 {task_id}에서 치명적 오류 발생!")
            raise ValueError("잘못된 데이터 형식입니다.")

        print(f"✅ [성공] 작업 {task_id} 완료")
        return f"데이터 {task_id}"

    # TaskGroup 내의 다른 태스크가 실패하면, 남은 태스크들에게 CancelledError가 던져집니다.
    except asyncio.CancelledError:
        print(f"🛑 [취소] 작업 {task_id} 강제 중단됨 (동료 작업의 실패로 인한 연쇄 취소)")
        raise  # 취소 신호는 삼키지 않고 다시 던지는 것이 관례입니다.

async def main_taskgroup():
    print("=== TaskGroup 테스트 시작 ===")
    try:
        # Python 3.11+: 안전하게 여러 태스크를 묶어서 관리
        async with asyncio.TaskGroup() as tg:
            tg.create_task(fetch_data(1)) # 정상 처리 예정
            tg.create_task(fetch_data(2)) # 에러 발생 예정
            tg.create_task(fetch_data(3)) # 정상 처리 예정

    # Python 3.11+: ExceptionGroup을 처리하기 위한 except* 문법
    except* ValueError as eg:
        print(f"\n🚨 [예외 포착] 그룹 내에서 발생한 에러들: {eg.exceptions}")

# Jupyter/Colab에서는 await main_taskgroup() 로 실행하세요.
if __name__ == "__main__":
    # asyncio.run(main_taskgroup())
    await main_taskgroup()


=== TaskGroup 테스트 시작 ===
▶️ [시작] 작업 1 실행 중...
▶️ [시작] 작업 2 실행 중...
▶️ [시작] 작업 3 실행 중...
✅ [성공] 작업 1 완료
💥 [에러] 작업 2에서 치명적 오류 발생!
✅ [성공] 작업 3 완료

🚨 [예외 포착] 그룹 내에서 발생한 에러들: (ValueError('잘못된 데이터 형식입니다.'),)


In [15]:
# 무한 대기 방지 (Timeout)
import asyncio, time

async def slow_network_call():
    print("📡 서버에 데이터를 요청합니다... (최대 10초 소요 예정)")
    await asyncio.sleep(10) # 10초가 걸리는 느린 작업 시뮬레이션
    return "서버 응답 데이터"

async def safe_fetch():
    print("\n=== Timeout 테스트 시작 ===")
    start_time = time.perf_counter()

    try:
        # Python 3.11+: 특정 블록 전체에 대한 타임아웃 설정
        print("⏳ 5초 타임아웃 제한을 걸고 대기합니다.")
        async with asyncio.timeout(5.0):
            result = await slow_network_call()
            print(f"✅ 결과 수신: {result}")

    except TimeoutError:
        print("🚨 [타임아웃 에러] 5초 안에 응답을 받지 못해 연결을 강제 종료합니다.")

    finally:
        elapsed = time.perf_counter() - start_time
        print(f"⏱️ 실제 대기한 총 시간: {elapsed:.2f}초") # 약 5.00초 출력됨

# Jupyter/Colab에서는 await safe_fetch() 로 실행하세요.
if __name__ == "__main__":
    # asyncio.run(safe_fetch())
    await safe_fetch()


=== Timeout 테스트 시작 ===
⏳ 5초 타임아웃 제한을 걸고 대기합니다.
📡 서버에 데이터를 요청합니다... (최대 10초 소요 예정)
🚨 [타임아웃 에러] 5초 안에 응답을 받지 못해 연결을 강제 종료합니다.
⏱️ 실제 대기한 총 시간: 5.01초
